# Re-run Step 5 export with the fixed Model B Integrated Gradients (issue #15/#17)

`model_b_phase4_projected_norm_attr_36bp` in `model_analysis_summary.json`
was all-zero for every Case Study - `colab_verify_model_b_ig_fix.ipynb`
already confirmed the root cause and fix (attributing against the
pre-encoder embedding tensor and replaying the full encoder, instead of
backpropagating only through `EsmClassificationHead`'s CLS-only pooling;
22/22 `test_integrated_gradients.py` tests pass on GPU).

This notebook re-runs the full `step5_export.py` pipeline with the fix in
place, to regenerate `model_analysis_summary.json` +
`model_analysis_arrays.npz` with correct Case Study IG values for Model B.
Runs on Colab rather than locally because this machine's ~8GB RAM was not
enough headroom for backprop through the NT encoder's ~500M parameters
(see `_frozen_parameters` in `integrated_gradients.py` for the mitigation
that's in place regardless - this still needs more RAM than local has free
alongside Model A + Model B_Predictor + Model B_XAIPredictor all loaded at
once).


## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Point at your uploaded assets

Edit `DRIVE_DIR` if needed. Everything `step5_export.py` reads from the
working directory must be here - upload anything missing before
continuing (the small json/npz files are already in your local `xai_test/`
repo checkout; the model directories/weights were already needed by prior
Colab notebooks in this project).


In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/xai_test"  # <-- EDIT ME

required = [
    "NT_sacas9_fintuned_model",
    "hybrid_xgb_model.json",
    "best_model_fold1.pth",
    "test_metadata.csv",
    "ism_delta.npz",
    "mismatch_delta.npz",
    "mismatch_complex_metadata.json",
    "ablation_results.json",
    "shap_results.json",
    "shap_values.npz",
    "attention_rollout_matrix.npz",
    "attention_rollout_summary.json",
]
missing = [
    name for name in required if not os.path.exists(os.path.join(DRIVE_DIR, name))
]
if missing:
    raise FileNotFoundError(
        f"Missing in {DRIVE_DIR}: {missing}. Upload these to Drive first, or fix DRIVE_DIR."
    )

os.chdir(DRIVE_DIR)
print("Working directory:", os.getcwd())

## 3. Install ViennaRNA CLI (RNAfold / RNAduplex) and Python deps

In [ ]:
!apt-get -qq update && apt-get -qq install -y vienna-rna
!which RNAfold && which RNAduplex

In [ ]:
!pip install -q transformers biopython xgboost joblib

## 4. Write the predictor + analysis modules

Mirrors the current committed `xai_test/` repo exactly (embedded at
notebook-build time, not fetched at runtime) - includes the Model B IG
fix (`model_b_xai_wrapper.py` / `integrated_gradients.py`).


In [ ]:
%%writefile model_a_wrapper.py
import numpy as np
import torch
from torch import nn


class OneHotAdapter(nn.Module):
    def __init__(self, input_dim: int = 4, hidden_dim: int = 128):
        super().__init__()
        # 가중치 형태 [128, 4]에 맞게 nn.Linear로 변경
        self.proj = nn.Linear(input_dim, hidden_dim)

    def forward(self, x):
        # x shape: [Batch, 36, 4]
        x = self.proj(x)  # -> [Batch, 36, 128]
        return x.transpose(1, 2)  # CNN 백본 입력에 맞게 변경 -> [Batch, 128, 36]


class CNN_RNN_Backbone(nn.Module):
    def __init__(
        self,
        input_dim: int = 128,
        conv_channels: int = 64,
        kernel_size: int = 3,
        lstm_hidden: int = 128,
        lstm_layers: int = 1,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.conv = nn.Conv1d(
            input_dim, conv_channels, kernel_size, padding=kernel_size // 2
        )
        self.act = nn.ReLU()
        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden),
            nn.ReLU(),
            nn.Linear(lstm_hidden, 1),
        )

    def forward(self, x):
        # x shape: [Batch, 128, 36]
        x = self.act(self.conv(x))
        x = x.transpose(1, 2)  # [Batch, 36, 64]
        out, _ = self.lstm(x)  # out shape: [Batch, 36, 128]

        # 마지막 시점(sequence)의 hidden state를 추출하여 분류기 통과
        out = self.head(out[:, -1, :])
        return out


class IntegratedPredictor(nn.Module):
    def __init__(self, adapter: nn.Module, backbone: nn.Module):
        super().__init__()
        self.adapter = adapter
        self.backbone = backbone

    def forward(self, x):
        x = self.adapter(x)
        x = self.backbone(x)
        return x.squeeze(-1)  # 1D Array 변환


class Model_A_Predictor:
    def __init__(self, weight_path: str | None = None, dropout_rate: float = 0.0):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # 원본 구조와 동일하게 어댑터와 백본 조립
        adapter = OneHotAdapter(input_dim=4, hidden_dim=128)
        backbone = CNN_RNN_Backbone(
            input_dim=128, lstm_hidden=128, dropout=dropout_rate
        )
        self.model = IntegratedPredictor(adapter=adapter, backbone=backbone).to(
            self.device
        )

        # 가중치 로드
        if weight_path:
            self.model.load_state_dict(
                torch.load(weight_path, map_location=self.device)
            )
        self.model.eval()

        self.nuc_map = {
            "A": [1.0, 0.0, 0.0, 0.0],
            "C": [0.0, 1.0, 0.0, 0.0],
            "G": [0.0, 0.0, 1.0, 0.0],
            "T": [0.0, 0.0, 0.0, 1.0],
        }

    def encode_one_hot(self, sequences: list[str]) -> torch.Tensor:
        encoded_list = []
        for seq in sequences:
            encoded = [self.nuc_map.get(n.upper(), [0.0, 0.0, 0.0, 0.0]) for n in seq]
            encoded_list.append(encoded)
        tensor = torch.tensor(encoded_list, dtype=torch.float32)
        return tensor  # shape: [Batch, 36, 4]

    def predict(self, sequences: list[str]) -> np.ndarray:
        x = self.encode_one_hot(sequences).to(self.device)
        with torch.no_grad():
            preds = self.model(x).cpu().numpy()
        return preds


In [ ]:
%%writefile model_b_wrapper.py
import os
import pickle
import re
import subprocess

import joblib
import numpy as np
import torch
import xgboost as xgb
from Bio.SeqUtils import MeltingTemp as mt
from Bio.SeqUtils import gc_fraction
from transformers import AutoModelForMaskedLM, AutoTokenizer


class Model_B_Predictor:
    def __init__(self, nt_model_dir: str, xgb_model_path: str):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Loading NT backbone from {nt_model_dir}...")
        self.tokenizer = AutoTokenizer.from_pretrained(nt_model_dir)
        self.nt_model = AutoModelForMaskedLM.from_pretrained(nt_model_dir).to(
            self.device
        )
        self.nt_model.eval()

        print(f"Loading XGBoost model from {xgb_model_path}...")
        self.xgb_model = self._load_xgb_model(xgb_model_path)

    def _load_xgb_model(self, path: str):
        if not os.path.exists(path):
            raise FileNotFoundError(f"XGBoost model file not found: {path}")

        try:
            with open(path, "rb") as f:
                return pickle.load(f)
        except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
            try:
                return joblib.load(path)
            except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
                model = xgb.XGBRegressor()
                model.load_model(path)
                return model

    def extract_nt_embeddings(self, sequences: list[str]) -> np.ndarray:
        inputs = self.tokenizer(
            sequences, return_tensors="pt", padding=True, truncation=True
        ).to(self.device)
        with torch.no_grad():
            outputs = self.nt_model(**inputs, output_hidden_states=True)
            last_hidden_state = outputs.hidden_states[-1]

            batch_size, seq_len, hidden_dim = last_hidden_state.shape
            assert seq_len == 7, f"Expected 7 tokens, got {seq_len}"
            assert hidden_dim == 1280, f"Expected 1280 hidden dim, got {hidden_dim}"

            flattened_embeddings = last_hidden_state.view(batch_size, -1).cpu().numpy()
        return flattened_embeddings

    def _get_guide_rna(self, dna_seq: str) -> str:
        complement_map = str.maketrans("ATCGatcg", "UAGCuagc")
        return dna_seq.translate(complement_map)[::-1]

    # RNAfold/RNAduplex are external processes: each subprocess.run() call pays
    # OS process-spawn overhead regardless of how little work it does. Both CLIs
    # accept many records over a single stdin stream, so batching sequences into
    # one call (chunked, to keep any single call's stdin/stdout bounded) turns an
    # O(n) process-spawn cost into O(n / batch_size) - the dominant cost at
    # ISM-sweep scale (tens of thousands of mutants). Deliberate scope override of
    # issue #9's "RNAfold/RNAduplex call batching" Out-of-Scope note, requested
    # directly by the user; verified to reproduce the exact same per-sequence
    # energy values as the prior one-process-per-sequence implementation.
    PHYSICAL_FEATURE_BATCH_SIZE = 500

    @staticmethod
    def _chunked(items: list, size: int):
        for i in range(0, len(items), size):
            yield items[i : i + size]

    def _calc_mfe_batch(self, seqs: list[str]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(seqs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAfold CLI 호출 (--noPS: 이미지 생성 방지)
            proc = subprocess.run(
                ["RNAfold", "--noPS"],
                input="".join(f"{seq}\n" for seq in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # Each record prints "<sequence>\n<dot-bracket structure> (<energy>)\n".
            # The structure diagram's own parentheses never have a digit directly
            # inside them, so this regex matches exactly one energy per record,
            # in input order. 출력 예시: ... ( -4.20)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAfold batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def _calc_duplex_dg_batch(self, pairs: list[tuple[str, str]]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(pairs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAduplex CLI 호출
            proc = subprocess.run(
                ["RNAduplex"],
                input="".join(f"{guide}\n{target}\n" for guide, target in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # 출력 예시: ... :  1,36 :   1,36 (-32.50)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAduplex batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def compute_physical_features(self, sequences: list[str]) -> np.ndarray:
        # 1. MFE (RNAfold.exe), batched across all sequences
        mfe_values = self._calc_mfe_batch(sequences)

        # 2. Duplex ΔG (RNAduplex.exe), batched across all sequences
        guide_rnas = [self._get_guide_rna(seq) for seq in sequences]
        dg_values = self._calc_duplex_dg_batch(list(zip(guide_rnas, sequences)))

        features = []
        for seq, mfe, dg in zip(sequences, mfe_values, dg_values):
            # 3. Tm (must match training-time nn_table=DNA_NN4 in nt_feature_stacking.py)
            tm = mt.Tm_NN(seq, nn_table=mt.DNA_NN4)

            # 4. GC Content
            gc = gc_fraction(seq) * 100.0

            features.append([mfe, dg, tm, gc])
        return np.array(features, dtype=np.float32)

    def predict(self, sequences: list[str]) -> np.ndarray:
        embeddings = self.extract_nt_embeddings(sequences)
        physical_feats = self.compute_physical_features(sequences)

        final_features = np.hstack([embeddings, physical_feats])
        assert final_features.shape[1] == 8964, (
            f"Dimension mismatch: Expected 8964, got {final_features.shape[1]}"
        )

        preds = self.xgb_model.predict(final_features)
        return np.array(preds).flatten()


In [ ]:
%%writefile model_b_xai_wrapper.py
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


class Model_B_XAIPredictor:
    """XAI-only counterpart to `Model_B_Predictor` (see ADR-0001).

    Loads the same NT checkpoint via `AutoModelForSequenceClassification`
    (not `AutoModelForMaskedLM`) so gradients flow through the checkpoint's
    own regression head and `output_attentions=True` is available. Has no
    analysis behavior of its own; Integrated Gradients and Attention Rollout
    build on top of it. Must never modify or risk `Model_B_Predictor`'s
    production inference path.
    """

    EXPECTED_SEQ_LEN = 7
    EXPECTED_HIDDEN_DIM = 1280

    def __init__(self, nt_model_dir: str):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(
            f"Loading NT backbone (XAI, sequence-classification head) from {nt_model_dir}..."
        )
        self.tokenizer = AutoTokenizer.from_pretrained(nt_model_dir)
        # "eager" attention is required for output_attentions=True; the default
        # "sdpa" backend silently returns no attention weights instead.
        self.nt_model = AutoModelForSequenceClassification.from_pretrained(
            nt_model_dir, attn_implementation="eager"
        ).to(self.device)
        self.nt_model.eval()

    def _tokenize(self, sequences: list[str]):
        return self.tokenizer(
            sequences, return_tensors="pt", padding=True, truncation=True
        ).to(self.device)

    def get_token_embeddings(self, sequences: list[str]) -> torch.Tensor:
        """Return the last-layer token embeddings, gradients enabled.

        Shape is asserted here, before any consumer (e.g. Integrated
        Gradients) uses the tensor, so a malformed input raises immediately
        instead of silently propagating a wrong shape downstream.
        """
        inputs = self._tokenize(sequences)
        outputs = self.nt_model(**inputs, output_hidden_states=True)
        embeddings = outputs.hidden_states[-1]

        _batch_size, seq_len, hidden_dim = embeddings.shape
        assert seq_len == self.EXPECTED_SEQ_LEN, (
            f"Expected {self.EXPECTED_SEQ_LEN} tokens, got {seq_len}"
        )
        assert hidden_dim == self.EXPECTED_HIDDEN_DIM, (
            f"Expected {self.EXPECTED_HIDDEN_DIM} hidden dim, got {hidden_dim}"
        )
        assert embeddings.requires_grad, (
            "Embeddings must carry gradients for Integrated Gradients to run"
        )

        return embeddings

    def get_input_embeddings(
        self, sequences: list[str]
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Return (pre-encoder input embeddings, attention_mask), gradients enabled.

        This - not `get_token_embeddings`'s *post-encoder* last hidden
        state - is the point Integrated Gradients must attribute against:
        `EsmClassificationHead` pools only the CLS token's final hidden
        state (`features[:, 0, :]`), so every non-CLS post-encoder
        embedding is a dead end the head never reads - its IG attribution
        would be exactly zero by construction, not just small. Attributing
        against the embedding-lookup output instead lets gradients flow
        back through every self-attention layer, where non-CLS tokens do
        influence the CLS token's final representation.
        """
        inputs = self._tokenize(sequences)
        embeddings = self.nt_model.esm.embeddings(
            input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]
        )

        _batch_size, seq_len, hidden_dim = embeddings.shape
        assert seq_len == self.EXPECTED_SEQ_LEN, (
            f"Expected {self.EXPECTED_SEQ_LEN} tokens, got {seq_len}"
        )
        assert hidden_dim == self.EXPECTED_HIDDEN_DIM, (
            f"Expected {self.EXPECTED_HIDDEN_DIM} hidden dim, got {hidden_dim}"
        )
        assert embeddings.requires_grad, (
            "Embeddings must carry gradients for Integrated Gradients to run"
        )

        return embeddings, inputs["attention_mask"]

    def classify_from_input_embeddings(
        self, embeddings: torch.Tensor, attention_mask: torch.Tensor
    ) -> torch.Tensor:
        """Run the full encoder + classification head from pre-encoder embeddings.

        Unlike a head-only shortcut, this replays every self-attention
        layer so gradients reach every token position, not just CLS - see
        `get_input_embeddings`.
        """
        return self.nt_model(
            inputs_embeds=embeddings, attention_mask=attention_mask
        ).logits.squeeze(-1)

    def get_attentions(self, sequences: list[str]) -> tuple[torch.Tensor, ...]:
        """Return per-layer attention matrices, `<cls>` still included.

        Stripping `<cls>` and mapping to nucleotide coordinates happens
        downstream, in Attention Rollout.
        """
        inputs = self._tokenize(sequences)
        outputs = self.nt_model(**inputs, output_attentions=True)
        return outputs.attentions


In [ ]:
%%writefile ism_sweep.py
"""In-silico saturation mutagenesis (ISM) sweep (Step 3, see CONTEXT.md / plan_realize.md).

Generates every position x alternative-base substitution for a set of 36bp
Testset sequences, scores wild-type and mutants through a predictor's own
`predict()`, and reduces the result to a relative delta-score sensitivity
array. Works against any predictor exposing `predict(sequences) -> array`
(duck-typed to `Model_A_Predictor` / `Model_B_Predictor`), so Model B's
per-mutant physical-feature recomputation (MFE, dG, Tm, GC) happens for free:
every mutant is dispatched as a full sequence, never a cached wild-type value
or a bare delta.
"""

from collections.abc import Sequence

import numpy as np

BASES = ("A", "C", "G", "T")

# Relative delta-Score is undefined (unbounded) as Score_WT -> 0; scores at or
# below this magnitude are masked to NaN instead of producing a spurious ratio.
RELATIVE_DELTA_NAN_THRESHOLD = 1e-4

# Neither Model_A_Predictor nor Model_B_Predictor batches internally - predict()
# runs whatever list it's given as one forward pass. An ISM sweep can hand it
# tens of thousands of mutants at once, which is large enough to exhaust GPU
# memory (observed: CUDA OOM inside Model A's LSTM forward pass on a 14.56GB
# T4 trying to allocate 20.71 GiB for one such call). Chunking dispatch here
# keeps peak memory bounded regardless of sweep size, without changing either
# predictor's own predict() contract.
DEFAULT_PREDICT_BATCH_SIZE = 256


def alt_bases_for(wt_base: str) -> list[str]:
    """The 3 non-wild-type bases, in fixed lexicographic order."""
    wt_base = wt_base.upper()
    if wt_base not in BASES:
        raise ValueError(f"Not a valid base: {wt_base!r}")
    return sorted(set(BASES) - {wt_base})


def generate_ism_mutants(sequence: str) -> list[str]:
    """All position x alternative-base substitutions for `sequence`.

    Ordered position-major (0..len-1), alternative-base lexicographic-minor,
    matching the `ism_delta` array's (position, alternative_base) axes.
    """
    sequence = sequence.upper()
    mutants = []
    for pos, wt_base in enumerate(sequence):
        for alt_base in alt_bases_for(wt_base):
            mutants.append(sequence[:pos] + alt_base + sequence[pos + 1 :])
    return mutants


def dispatch_in_batches(
    fn, items: list[str], batch_size: int, stack=np.concatenate
) -> np.ndarray:
    """Call `fn()` in bounded chunks and combine the results with `stack`.

    Generic version of the chunking `predict_in_batches` needs for 1D score
    arrays; `step4_ablation.py` reuses this directly (with `stack=np.vstack`)
    to batch `Model_B_Predictor.extract_nt_embeddings()`'s 2D output the same
    way, instead of re-deriving the chunk loop.
    """
    if not items:
        return np.array([])
    chunks = [fn(items[i : i + batch_size]) for i in range(0, len(items), batch_size)]
    return stack(chunks)


def predict_in_batches(predictor, sequences: list[str], batch_size: int) -> np.ndarray:
    """Dispatch `predictor.predict()` in bounded chunks (shared with mismatch_profiling.py)."""
    if not sequences:
        return np.array([], dtype=np.float64)

    def _predict_chunk(chunk):
        return np.asarray(predictor.predict(chunk), dtype=np.float64)

    return dispatch_in_batches(
        _predict_chunk, sequences, batch_size, stack=np.concatenate
    )


def relative_delta(
    wt_scores: np.ndarray, mutant_scores: np.ndarray, nan_threshold: float
) -> np.ndarray:
    """`(mutant - wt) / |wt|` per sample, NaN-masked wherever `|wt| <= nan_threshold`.

    `wt_scores` is 1D (one value per sample); `mutant_scores` broadcasts against
    it along its leading axis - shape `(n, positions, alt_bases)` for the ISM
    sweep, `(n, scenarios)` for complex mismatch profiling. Shared so both
    modules apply the exact same guardrail (see `RELATIVE_DELTA_NAN_THRESHOLD`).
    """
    broadcast_shape = (wt_scores.shape[0],) + (1,) * (mutant_scores.ndim - 1)
    wt_broadcast = wt_scores.reshape(broadcast_shape)
    with np.errstate(divide="ignore", invalid="ignore"):
        delta = (mutant_scores - wt_broadcast) / np.abs(wt_broadcast)
    delta[np.abs(wt_scores) <= nan_threshold] = np.nan
    return delta


def compute_ism_delta(
    sequences: Sequence[str],
    predictor,
    nan_threshold: float = RELATIVE_DELTA_NAN_THRESHOLD,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> np.ndarray:
    """Run the full ISM sweep for one predictor.

    Returns `ism_delta`, shape `(len(sequences), seq_len, 3)`: the relative
    score delta `(Score_mutant - Score_WT) / |Score_WT|` per position x
    lexicographic alternative base, NaN-masked wherever `|Score_WT| <=
    nan_threshold`. `predict_batch_size` bounds how many sequences reach
    `predictor.predict()` per call; lower it if you still hit an
    out-of-memory error.
    """
    sequences = list(sequences)
    seq_len = len(sequences[0])
    if any(len(seq) != seq_len for seq in sequences):
        raise ValueError("All sequences must share the same length for the ISM sweep.")

    wt_scores = predict_in_batches(predictor, sequences, predict_batch_size)

    mutants: list[str] = []
    for sequence in sequences:
        mutants.extend(generate_ism_mutants(sequence))

    mutant_scores = predict_in_batches(predictor, mutants, predict_batch_size)
    mutant_scores = mutant_scores.reshape(len(sequences), seq_len, 3)

    return relative_delta(wt_scores, mutant_scores, nan_threshold)


def run_ism_sweep(
    sequences: Sequence[str],
    predictor_a,
    predictor_b,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> dict[str, np.ndarray]:
    """Run the ISM sweep for both models over the same Testset sequences."""
    return {
        "ism_delta_model_a": compute_ism_delta(
            sequences, predictor_a, predict_batch_size=predict_batch_size
        ),
        "ism_delta_model_b": compute_ism_delta(
            sequences, predictor_b, predict_batch_size=predict_batch_size
        ),
    }


In [ ]:
%%writefile mismatch_profiling.py
"""Mismatch tolerance profiling: single + complex scenarios (Step 3, issue #12).

Single mismatch: a single mismatch *is* a single ISM mutation, so
`compute_mismatch_single` reuses `ism_sweep.compute_ism_delta` unchanged for
`mismatch_single_delta`, and reduces it to the worst-case (minimum relative
delta) per position for `mismatch_single_min`.

Complex mismatch: a fixed set of 15 reproducible (`random_seed=42`)
multi-position (1-3bp) scenarios - 5 Seed-region, 5 Distal-region, 5
Intermittent - applied identically (same positions, same per-position
lexicographic-alternate-base selection, see `alt_bases_for`) to every Testset
sequence. Substitutions stay self-relative to each sequence's own base at
each position (the same convention `ism_sweep` uses), which guarantees every
substitution is a genuine mismatch regardless of the underlying sequence,
while keeping one scenario definition comparable ("profiled") across the
whole population. Region boundaries: Seed = positions 17-24 (the 8bp of the
protospacer immediately 5' of - i.e. PAM-proximal to - the confirmed PAM at
25-30, see integrated_gradients.py's PAM_REGION_START/END), Distal =
positions 0-7 (the farthest 8bp from the PAM), Intermittent = non-consecutive
positions scattered across the full 0-35 span.

Earlier versions of this module (issue #12) fixed Seed at positions 0-7,
before the PAM's exact position was pinned down (issue #15 confirmed
25-30). That range was mislabeled "PAM-adjacent" - positions 0-7 are in fact
the farthest possible point from the PAM - and the old "Distal" range
(28-35) actually overlapped the PAM itself. Corrected here so "Seed" is
biologically PAM-proximal and "Distal" is biologically PAM-distal, matching
plan.md/plan_realize.md's stated hypothesis that cleavage sensitivity rises
closer to the PAM.

`ref_bases`/`alt_bases` in a scenario's exported metadata are illustrative:
computed against one `example_sequence` (by convention, Testset sample_id 0)
because the literal ref base at a scenario's positions differs per sequence -
the same position + alt-base-index rule is applied to every sequence's own
base, exactly as ISM does.
"""

from collections.abc import Sequence
from dataclasses import dataclass

import numpy as np
from ism_sweep import (
    DEFAULT_PREDICT_BATCH_SIZE,
    RELATIVE_DELTA_NAN_THRESHOLD,
    alt_bases_for,
    compute_ism_delta,
    predict_in_batches,
    relative_delta,
)

SEQ_LEN = 36
# PAM-proximal 8bp of the protospacer, immediately 5' of the confirmed PAM
# (25-30, see integrated_gradients.PAM_REGION_START/END).
SEED_REGION_START, SEED_REGION_END = 17, 25  # positions 17-24
DISTAL_REGION_START, DISTAL_REGION_END = 0, 8  # positions 0-7, farthest from the PAM
SCENARIOS_PER_REGION = 5
COMPLEX_MISMATCH_SEED = 42


@dataclass(frozen=True)
class MismatchScenario:
    """One fixed complex-mismatch scenario, applied identically across the Testset."""

    mutation_id: str
    region: str
    positions: tuple[int, ...]
    alt_base_indices: tuple[int, ...]  # index into alt_bases_for(wt) per position
    description: str


def _draw_alt_indices(rng: np.random.Generator, n: int) -> tuple[int, ...]:
    return tuple(int(x) for x in rng.integers(0, 3, size=n))


def _generate_consecutive_scenarios(
    rng: np.random.Generator, region: str, region_start: int, region_end: int
) -> list[MismatchScenario]:
    scenarios = []
    region_prefix = region.lower()
    seen_positions: set[tuple[int, ...]] = set()
    for _ in range(SCENARIOS_PER_REGION):
        while True:
            length = int(rng.integers(1, 4))  # 1..3 bp, per plan.md
            max_start = region_end - length
            start = int(rng.integers(region_start, max_start + 1))
            positions = tuple(range(start, start + length))
            if positions not in seen_positions:
                seen_positions.add(positions)
                break
        alt_base_indices = _draw_alt_indices(rng, length)
        pos_label = "_".join(str(p) for p in positions)
        scenarios.append(
            MismatchScenario(
                mutation_id=f"{region_prefix}_consecutive_{length}bp_pos{pos_label}",
                region=region,
                positions=positions,
                alt_base_indices=alt_base_indices,
                description=f"{region} consecutive {length}bp mismatch at position(s) {list(positions)}",
            )
        )
    return scenarios


def _generate_intermittent_scenarios(
    rng: np.random.Generator,
) -> list[MismatchScenario]:
    scenarios = []
    seen_positions: set[tuple[int, ...]] = set()
    for _ in range(SCENARIOS_PER_REGION):
        while True:
            count = int(rng.integers(2, 4))  # 2..3 non-consecutive positions
            positions = tuple(
                sorted(int(x) for x in rng.choice(SEQ_LEN, size=count, replace=False))
            )
            is_consecutive = all(
                positions[i + 1] - positions[i] == 1 for i in range(len(positions) - 1)
            )
            if not is_consecutive and positions not in seen_positions:
                seen_positions.add(positions)
                break
        alt_base_indices = _draw_alt_indices(rng, count)
        pos_label = "_".join(str(p) for p in positions)
        scenarios.append(
            MismatchScenario(
                mutation_id=f"intermittent_nonconsecutive_{count}bp_pos{pos_label}",
                region="Intermittent",
                positions=positions,
                alt_base_indices=alt_base_indices,
                description=f"Intermittent non-consecutive {count}bp mismatch at position(s) {list(positions)}",
            )
        )
    return scenarios


def generate_mismatch_complex_scenarios(
    seed: int = COMPLEX_MISMATCH_SEED,
) -> list[MismatchScenario]:
    """The fixed 15 complex mismatch scenarios: 5 Seed + 5 Distal + 5 Intermittent.

    Fully determined by `seed` - drawn in this order (Seed, then Distal, then
    Intermittent) from one `np.random.default_rng(seed)` stream, so the same
    seed always reproduces the same 15 scenarios.
    """
    rng = np.random.default_rng(seed)
    scenarios = []
    scenarios.extend(
        _generate_consecutive_scenarios(rng, "Seed", SEED_REGION_START, SEED_REGION_END)
    )
    scenarios.extend(
        _generate_consecutive_scenarios(
            rng, "Distal", DISTAL_REGION_START, DISTAL_REGION_END
        )
    )
    scenarios.extend(_generate_intermittent_scenarios(rng))
    return scenarios


def apply_scenario(scenario: MismatchScenario, sequence: str) -> str:
    """Apply `scenario`'s positions/alt-indices to `sequence`'s own bases."""
    chars = list(sequence.upper())
    for pos, alt_idx in zip(scenario.positions, scenario.alt_base_indices):
        chars[pos] = alt_bases_for(chars[pos])[alt_idx]
    return "".join(chars)


def scenario_to_metadata(
    scenario: MismatchScenario, scenario_index: int, example_sequence: str
) -> dict:
    """Export metadata for one scenario (see module docstring on ref/alt bases)."""
    example_sequence = example_sequence.upper()
    ref_bases = [example_sequence[p] for p in scenario.positions]
    alt_bases = [
        alt_bases_for(example_sequence[p])[idx]
        for p, idx in zip(scenario.positions, scenario.alt_base_indices)
    ]
    return {
        "scenario_index": scenario_index,
        "mutation_id": scenario.mutation_id,
        "region": scenario.region,
        "positions": list(scenario.positions),
        "ref_bases": ref_bases,
        "alt_bases": alt_bases,
        "description": scenario.description,
    }


def compute_mismatch_single(
    sequences: Sequence[str],
    predictor,
    nan_threshold: float = RELATIVE_DELTA_NAN_THRESHOLD,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> tuple[np.ndarray, np.ndarray]:
    """Single mismatch profiling for one predictor.

    Returns `(mismatch_single_delta, mismatch_single_min)`:
    - `mismatch_single_delta`, shape `(len(sequences), seq_len, 3)`: identical
      to `ism_sweep.compute_ism_delta` (a single mismatch is a single ISM
      mutation) - all 3 substitutions preserved per position.
    - `mismatch_single_min`, shape `(len(sequences), seq_len)`: the worst-case
      (minimum) relative delta across the 3 alternatives at each position.
    """
    mismatch_single_delta = compute_ism_delta(
        sequences,
        predictor,
        nan_threshold=nan_threshold,
        predict_batch_size=predict_batch_size,
    )
    mismatch_single_min = np.min(mismatch_single_delta, axis=-1)
    return mismatch_single_delta, mismatch_single_min


def compute_mismatch_complex_delta(
    sequences: Sequence[str],
    predictor,
    scenarios: list[MismatchScenario] | None = None,
    nan_threshold: float = RELATIVE_DELTA_NAN_THRESHOLD,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> np.ndarray:
    """Complex mismatch profiling for one predictor.

    Returns `mismatch_complex`, shape `(len(sequences), len(scenarios))`: the
    relative score delta `(Score_mutant - Score_WT) / |Score_WT|` per
    scenario, NaN-masked wherever `|Score_WT| <= nan_threshold` (same
    guardrail as the ISM sweep).
    """
    if scenarios is None:
        scenarios = generate_mismatch_complex_scenarios()

    sequences = list(sequences)
    seq_len = len(sequences[0])
    if any(len(seq) != seq_len for seq in sequences):
        raise ValueError(
            "All sequences must share the same length for mismatch profiling."
        )

    wt_scores = predict_in_batches(predictor, sequences, predict_batch_size)

    # Scenario-major, sequence-minor ordering, matching the reshape below.
    mutants: list[str] = []
    for scenario in scenarios:
        for sequence in sequences:
            mutants.append(apply_scenario(scenario, sequence))

    mutant_scores = predict_in_batches(predictor, mutants, predict_batch_size)
    mutant_scores = mutant_scores.reshape(len(scenarios), len(sequences)).T

    return relative_delta(wt_scores, mutant_scores, nan_threshold)


def run_mismatch_profiling(
    sequences: Sequence[str],
    predictor_a,
    predictor_b,
    scenarios: list[MismatchScenario] | None = None,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> dict[str, np.ndarray]:
    """Run single + complex mismatch profiling for both models over the same Testset."""
    if scenarios is None:
        scenarios = generate_mismatch_complex_scenarios()

    result: dict[str, np.ndarray] = {}
    for model_name, predictor in (("model_a", predictor_a), ("model_b", predictor_b)):
        single_delta, single_min = compute_mismatch_single(
            sequences, predictor, predict_batch_size=predict_batch_size
        )
        result[f"mismatch_single_delta_{model_name}"] = single_delta
        result[f"mismatch_single_min_{model_name}"] = single_min
        result[f"mismatch_complex_{model_name}"] = compute_mismatch_complex_delta(
            sequences,
            predictor,
            scenarios=scenarios,
            predict_batch_size=predict_batch_size,
        )
    return result


In [ ]:
%%writefile integrated_gradients.py
"""Integrated Gradients: Model A vs Model B sequence-level comparison (Step 2, issue #15).

Model A is attributed directly against its one-hot input layer with an
all-zero one-hot baseline. Model B is attributed against
`Model_B_XAIPredictor`'s `(batch, 7, 1280)` pre-encoder input-embedding
tensor (see ADR-0001 / issue #10) with a fixed zero-embedding baseline of
the same shape, running the integration through the *whole* encoder +
classification head (`classify_from_input_embeddings`). Attributing against
the *post-encoder* last hidden state instead (an earlier version of this
module did) is a dead end: `EsmClassificationHead` pools only the CLS
token's final hidden state (`features[:, 0, :]`), so every non-CLS
post-encoder embedding's gradient is exactly zero by construction - not
just small - regardless of the actual sequence. Attributing against the
pre-encoder embeddings and replaying the full encoder instead lets
gradients flow back through every self-attention layer, where non-CLS
tokens do influence the CLS token's final representation.

Model B's 6-mer token attributions are projected back to the 36bp sequence
via the Deterministic Projection Rule (CONTEXT.md): each token's attribution
divided evenly across the 6 nucleotides it spans. This is an explicit
analysis assumption, not a claim that the model perceives individual
nucleotides independently.

Both models' final attribution arrays are within-sequence L1-normalized
before any comparison - comparisons are relative-distribution based, never
raw magnitude - especially at the PAM (`NNGRRT`) and Seed (PAM-proximal
17-24bp) positions. Seed boundaries (17-24) reuse mismatch_profiling.py's
existing convention; PAM boundaries (25-30bp inclusive) are this dataset's
confirmed fixed sequence-construction alignment (36bp = ... + 8bp Seed at
17-24 + 6bp PAM at 25-30 + 5bp 3' flank at 31-35).
"""

import contextlib

import numpy as np
import torch
from ism_sweep import dispatch_in_batches
from mismatch_profiling import SEED_REGION_END, SEED_REGION_START
from model_a_wrapper import Model_A_Predictor
from model_b_xai_wrapper import Model_B_XAIPredictor

DEFAULT_IG_STEPS = 50

# Lower than ism_sweep.DEFAULT_PREDICT_BATCH_SIZE (256): unlike a single
# predict() forward pass, IG retains a full autograd graph per step across
# `steps` sequential backward passes, so it is far more memory-hungry per
# sequence - the same class of CUDA OOM ism_sweep.py's own batching guards
# against, just reached at a smaller batch size.
DEFAULT_IG_BATCH_SIZE = 32

CLS_TOKEN_COUNT = 1
TOKEN_NT_SPAN = (
    6  # 6-mer tokenizer: each non-CLS token covers 6 consecutive nucleotides
)

# PAM (NNGRRT), confirmed dataset alignment: 25-30 inclusive (25:31 as a slice).
PAM_REGION_START, PAM_REGION_END = 25, 31


@contextlib.contextmanager
def _frozen_parameters(module: torch.nn.Module):
    """Temporarily disable `requires_grad` on every parameter in `module`.

    Integrated Gradients only needs a gradient w.r.t. the interpolated
    *input* tensor - `backward()` still populates that regardless of
    whether `module`'s own weights require grad, since autograd tracks any
    graph touching a requires_grad leaf. Without this, backprop through
    Model B's ~500M-parameter NT encoder (unlike Model A's ~140K-parameter
    CNN+RNN) allocates a same-sized `.grad` buffer for every one of those
    weights at every interpolation step - pure waste that was enough to
    exhaust this machine's RAM once IG started replaying the full encoder
    (see `integrated_gradients_model_b_tokens`) instead of just its small
    classification head. Restores each parameter's original
    `requires_grad` afterward so other callers (e.g. `get_token_embeddings`'s
    gradient-flow assertion, Attention Rollout) see the model unchanged.
    """
    originally_required = [p.requires_grad for p in module.parameters()]
    for p in module.parameters():
        p.requires_grad_(False)
    try:
        yield
    finally:
        for p, required in zip(module.parameters(), originally_required):
            p.requires_grad_(required)


def _integrated_gradients(
    forward_fn, inputs: torch.Tensor, baseline: torch.Tensor, steps: int
) -> torch.Tensor:
    """Riemann-sum Integrated Gradients (Sundararajan et al., 2017).

    `forward_fn` must map a tensor shaped like `inputs` to a `(batch,)`
    scalar-per-sample output. Summing that output before `backward()` is
    safe because every sample's output depends only on its own slice of
    `inputs` (true for both Model A's per-sequence CNN+RNN forward pass and
    Model B's per-token classifier head) - the batch-summed gradient still
    equals each sample's own gradient, so one backward pass per step covers
    the whole batch instead of one per sample.

    Runs with cuDNN's fused RNN kernel disabled: that kernel only supports
    a backward pass when the forward ran in training mode, so Model A's
    eval-mode LSTM raises "cudnn RNN backward can only be called in
    training mode" on GPU otherwise (dropout=0 here makes train/eval mode
    otherwise identical, so this changes nothing except which kernel runs).
    Falls back to a slower, generic RNN backward that works in eval mode
    regardless - a no-op for Model B (no RNN involved) and on CPU (cuDNN
    never applies there), so this is safe to apply unconditionally.
    """
    diff = inputs - baseline
    accumulated_grad = torch.zeros_like(inputs)
    with torch.backends.cudnn.flags(enabled=False):
        for step in range(1, steps + 1):
            alpha = step / steps
            interpolated = (
                (baseline + alpha * diff).detach().clone().requires_grad_(True)
            )
            output = forward_fn(interpolated)
            output.sum().backward()
            accumulated_grad += interpolated.grad
    avg_grad = accumulated_grad / steps
    return (diff * avg_grad).detach()


def integrated_gradients_model_a(
    predictor: Model_A_Predictor,
    sequences: list[str],
    steps: int = DEFAULT_IG_STEPS,
    batch_size: int = DEFAULT_IG_BATCH_SIZE,
) -> np.ndarray:
    """Per-nucleotide IG attribution for Model A, shape `(batch, 36)`.

    Runs against Model A's one-hot input layer with an all-zero one-hot
    baseline. Since the baseline is zero and a one-hot input is zero
    everywhere except the observed base's channel, only that channel ever
    carries a nonzero `(x - baseline)` term - summing over the 4-channel
    axis collapses to the attribution of the base actually present at each
    position, discarding nothing. `sequences` is dispatched in
    `batch_size`-sized chunks (see `DEFAULT_IG_BATCH_SIZE`) to bound peak
    memory the same way `ism_sweep.py` bounds `predict()` calls.
    """

    def _attribute_chunk(chunk: list[str]) -> np.ndarray:
        inputs = predictor.encode_one_hot(chunk).to(predictor.device)
        baseline = torch.zeros_like(inputs)
        attributions = _integrated_gradients(predictor.model, inputs, baseline, steps)
        return attributions.sum(dim=-1).cpu().numpy()

    return dispatch_in_batches(_attribute_chunk, list(sequences), batch_size)


def integrated_gradients_model_b_tokens(
    xai_predictor: Model_B_XAIPredictor,
    sequences: list[str],
    steps: int = DEFAULT_IG_STEPS,
    batch_size: int = DEFAULT_IG_BATCH_SIZE,
) -> np.ndarray:
    """Raw per-token IG attribution for Model B, shape `(batch, 7, 1280)`, CLS included.

    Attributes against the `(batch, 7, 1280)` *pre-encoder* input-embedding
    tensor `Model_B_XAIPredictor.get_input_embeddings` exposes, using a
    fixed zero-embedding baseline of the same shape, backpropagating
    through `classify_from_input_embeddings` - the full encoder + head,
    not the head alone (see this module's docstring for why attributing
    against the *post-encoder* embedding instead would make every non-CLS
    token's attribution exactly zero by construction). `attention_mask` is
    fixed per sequence (padding structure, not a differentiable quantity)
    and passed through unchanged at every interpolation step. The
    embedding shape is validated before attribution runs; a mismatch
    raises here rather than silently attributing against a
    differently-shaped tensor. `get_input_embeddings` already asserts this
    on its own, but that assertion compiles out entirely under Python's
    `-O` flag - this check does not, so the guarantee holds independent of
    how the caller runs. `sequences` is dispatched in `batch_size`-sized
    chunks for the same reason `integrated_gradients_model_a` is. The
    actual backward loop runs under `_frozen_parameters` (see its
    docstring) so backprop through the full encoder doesn't allocate a
    `.grad` buffer for the encoder's own ~500M parameters at every step.
    """

    def _attribute_chunk(chunk: list[str]) -> np.ndarray:
        embeddings, attention_mask = xai_predictor.get_input_embeddings(chunk)
        embeddings = embeddings.detach()

        expected_shape = (
            len(chunk),
            Model_B_XAIPredictor.EXPECTED_SEQ_LEN,
            Model_B_XAIPredictor.EXPECTED_HIDDEN_DIM,
        )
        if tuple(embeddings.shape) != expected_shape:
            raise ValueError(
                f"Model B embedding shape {tuple(embeddings.shape)} does not match "
                f"expected {expected_shape}; refusing to attribute against a "
                "differently-shaped tensor."
            )

        baseline = torch.zeros_like(embeddings)

        def _forward(interpolated: torch.Tensor) -> torch.Tensor:
            return xai_predictor.classify_from_input_embeddings(
                interpolated, attention_mask
            )

        # Frozen only around the actual backward loop, not `get_input_embeddings`
        # above - that's a single forward pass (never calls `.backward()`, so
        # frozen-or-not costs nothing there) whose own internal assertion
        # requires a normal, grad-enabled parameter state to pass.
        with _frozen_parameters(xai_predictor.nt_model):
            attributions = _integrated_gradients(_forward, embeddings, baseline, steps)
        return attributions.cpu().numpy()

    return dispatch_in_batches(_attribute_chunk, list(sequences), batch_size)


def project_model_b_attributions_to_nucleotides(
    token_attributions: np.ndarray,
) -> np.ndarray:
    """Deterministic Projection Rule: 6-mer token attribution -> 36bp nucleotide attribution.

    `token_attributions` is `(batch, 7, 1280)` (CLS + 6 tokens, hidden
    dimension not yet collapsed). Collapses the hidden dimension by summing
    (mirroring Model A's one-hot-channel collapse), drops the CLS token,
    then divides each of the 6 remaining tokens' attribution evenly across
    the 6 nucleotides it spans - an explicit analysis assumption (see
    CONTEXT.md's Deterministic Projection Rule), not evidence the model
    perceives individual nucleotides independently. Conservation holds:
    each nucleotide block sums back to exactly its source token's
    attribution.
    """
    _batch, n_tokens, _hidden = token_attributions.shape
    if n_tokens != Model_B_XAIPredictor.EXPECTED_SEQ_LEN:
        raise ValueError(
            f"Expected {Model_B_XAIPredictor.EXPECTED_SEQ_LEN} tokens, got {n_tokens}"
        )

    per_token = token_attributions.sum(axis=-1)  # (batch, 7)
    per_kmer_token = per_token[:, CLS_TOKEN_COUNT:]  # drop CLS -> (batch, 6)

    return (
        np.repeat(per_kmer_token, TOKEN_NT_SPAN, axis=1) / TOKEN_NT_SPAN
    )  # (batch, 36)


def l1_normalize(attribution: np.ndarray) -> np.ndarray:
    """Within-sequence L1 normalization: each row's absolute values sum to 1.

    Model A and Model B's attribution magnitudes are on unrelated scales -
    comparisons are only meaningful as relative distributions (per
    plan_realize.md Step 2), never raw magnitude. All-zero rows stay
    all-zero (dividing by zero L1 mass would otherwise produce NaN) since
    there is no distribution to normalize.
    """
    l1_mass = np.abs(attribution).sum(axis=-1, keepdims=True)
    safe_mass = np.where(l1_mass == 0, 1.0, l1_mass)
    return attribution / safe_mass


def region_attribution_share(
    normalized_attribution: np.ndarray, start: int, end: int
) -> np.ndarray:
    """Fraction of a sequence's (already L1-normalized) attribution mass in `[start, end)`."""
    return np.abs(normalized_attribution[:, start:end]).sum(axis=-1)


def run_integrated_gradients(
    sequences: list[str],
    predictor_a: Model_A_Predictor,
    xai_predictor_b: Model_B_XAIPredictor,
    steps: int = DEFAULT_IG_STEPS,
    batch_size: int = DEFAULT_IG_BATCH_SIZE,
) -> dict:
    """Run IG for both models and the PAM/Seed relative-distribution comparison.

    Returns raw and L1-normalized per-nucleotide attribution arrays for each
    model, Model B's pre-projection per-token attribution (CLS dropped), and
    each model's PAM (25-30bp) / Seed (17-24bp) attribution-mass share - the
    relative-distribution comparison issue #15's AC4 calls for.
    """
    ig_a = integrated_gradients_model_a(
        predictor_a, sequences, steps=steps, batch_size=batch_size
    )
    ig_a_norm = l1_normalize(ig_a)

    ig_b_tokens = integrated_gradients_model_b_tokens(
        xai_predictor_b, sequences, steps=steps, batch_size=batch_size
    )
    ig_b = project_model_b_attributions_to_nucleotides(ig_b_tokens)
    ig_b_norm = l1_normalize(ig_b)

    return {
        "ig_model_a": ig_a,
        "ig_model_a_normalized": ig_a_norm,
        "ig_model_b_tokens": ig_b_tokens[:, CLS_TOKEN_COUNT:, :].sum(axis=-1),
        "ig_model_b": ig_b,
        "ig_model_b_normalized": ig_b_norm,
        "pam_share_model_a": region_attribution_share(
            ig_a_norm, PAM_REGION_START, PAM_REGION_END
        ),
        "pam_share_model_b": region_attribution_share(
            ig_b_norm, PAM_REGION_START, PAM_REGION_END
        ),
        "seed_share_model_a": region_attribution_share(
            ig_a_norm, SEED_REGION_START, SEED_REGION_END
        ),
        "seed_share_model_b": region_attribution_share(
            ig_b_norm, SEED_REGION_START, SEED_REGION_END
        ),
    }


In [ ]:
%%writefile shap_analysis.py
"""Tree SHAP analysis for production Model B-Full (Step 4-2, issue #14).

Computes exact per-feature Tree SHAP contributions via XGBoost's native
`pred_contribs` prediction mode - the same Tree SHAP algorithm the `shap`
package implements, built directly into the booster, so no extra dependency
is needed. Separates the top-20 sequence-embedding features from the 4 named
physical features (MFE, delta-G, Tm, GC) and reports an aggregate physical
SHAP contribution ratio, matching issue #14's acceptance criteria.
"""

import os

import numpy as np
import xgboost as xgb

EMBEDDING_DIM = 8960
PHYSICAL_DIM = 4
# Column order must match Model_B_Predictor.compute_physical_features's
# hstack order (mfe, dg, tm, gc) - see model_b_wrapper.py.
PHYSICAL_FEATURE_NAMES = ("MFE", "delta_G", "Tm", "GC")

DEFAULT_TOP_K = 20


def compute_tree_shap(model: xgb.XGBRegressor, X: np.ndarray) -> np.ndarray:
    """Exact per-feature Tree SHAP contributions for `X`, bias column dropped.

    `pred_contribs=True` returns one extra trailing column (the expected-value
    bias term) needed for additivity - `row.sum() + bias == raw prediction`.
    That bias isn't a feature contribution, so it's sliced off here; every
    downstream consumer works with a plain `(n_samples, n_features)` matrix.
    """
    booster = model.get_booster()
    dmatrix = xgb.DMatrix(X)
    contribs = booster.predict(dmatrix, pred_contribs=True)
    return contribs[:, :-1]


def split_shap_by_group(
    shap_values: np.ndarray,
    embedding_dim: int = EMBEDDING_DIM,
    physical_dim: int = PHYSICAL_DIM,
) -> dict[str, np.ndarray]:
    """Slice a `(n, embedding_dim + physical_dim)` SHAP matrix by feature group."""
    assert shap_values.shape[1] == embedding_dim + physical_dim, (
        f"Expected {embedding_dim + physical_dim} SHAP columns, got {shap_values.shape[1]}"
    )
    return {
        "embedding": shap_values[:, :embedding_dim],
        "physical": shap_values[:, embedding_dim:],
    }


def top_k_embedding_features(
    shap_embedding: np.ndarray, k: int = DEFAULT_TOP_K
) -> list[dict]:
    """Rank embedding features by mean |SHAP| across samples, descending.

    `feature_index` is local to the embedding sub-block (0..embedding_dim-1,
    e.g. 0-8959 in production), not the full 8,964-wide matrix - callers
    that need a global column index must add `embedding_dim` themselves.
    Caps at the number of available columns rather than raising when `k`
    exceeds the embedding width (relevant for tests against synthetic,
    lower-dimensional embeddings).
    """
    mean_abs = np.abs(shap_embedding).mean(axis=0)
    order = np.argsort(mean_abs)[::-1][:k]
    return [
        {"feature_index": int(i), "mean_abs_shap": float(mean_abs[i])} for i in order
    ]


def physical_feature_shap_summary(shap_physical: np.ndarray) -> list[dict]:
    """Mean |SHAP| per named physical feature, in `PHYSICAL_FEATURE_NAMES` order."""
    mean_abs = np.abs(shap_physical).mean(axis=0)
    return [
        {"feature_name": name, "mean_abs_shap": float(mean_abs[i])}
        for i, name in enumerate(PHYSICAL_FEATURE_NAMES)
    ]


def aggregate_physical_contribution_ratio(
    shap_values: np.ndarray, embedding_dim: int = EMBEDDING_DIM
) -> float:
    """`sum(|SHAP_physical|) / sum(|SHAP_all|)` - a proportion in [0, 1]."""
    total_abs = np.abs(shap_values).sum()
    physical_abs = np.abs(shap_values[:, embedding_dim:]).sum()
    return float(physical_abs / total_abs)


def dependence_plot_data(
    X_physical: np.ndarray,
    shap_physical: np.ndarray,
) -> dict[str, dict[str, np.ndarray]]:
    """Per physical feature: its raw values paired with their own SHAP column."""
    return {
        name: {
            "feature_value": X_physical[:, i],
            "shap_value": shap_physical[:, i],
        }
        for i, name in enumerate(PHYSICAL_FEATURE_NAMES)
    }


def save_dependence_plots(
    X_physical: np.ndarray,
    shap_physical: np.ndarray,
    output_dir: str,
) -> dict[str, str]:
    """Render one feature-value-vs-SHAP-value scatter plot per physical feature."""
    # Imported lazily so importing this module (e.g. for the pure array logic
    # above, as every test in test_shap_analysis.py does) never pays for
    # matplotlib or forces its Agg backend unless a plot is actually requested.
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    os.makedirs(output_dir, exist_ok=True)
    data = dependence_plot_data(X_physical, shap_physical)

    paths = {}
    for name, values in data.items():
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.scatter(values["feature_value"], values["shap_value"], s=10, alpha=0.6)
        ax.axhline(0, color="grey", linewidth=0.5)
        ax.set_xlabel(name)
        ax.set_ylabel("SHAP value")
        ax.set_title(f"Tree SHAP dependence: {name}")
        path = os.path.join(output_dir, f"dependence_{name}.png")
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        paths[name] = path
    return paths


def run_shap_analysis(
    model: xgb.XGBRegressor,
    X: np.ndarray,
    embedding_dim: int = EMBEDDING_DIM,
    physical_dim: int = PHYSICAL_DIM,
    top_k: int = DEFAULT_TOP_K,
    dependence_plot_dir: str | None = None,
) -> dict:
    """Run the full Tree SHAP analysis bundle against issue #14's acceptance criteria.

    Returns `top_embedding_features` (top-`top_k`, by mean |SHAP|),
    `physical_feature_shap` (all 4 named physical features), the aggregate
    `physical_contribution_ratio`, the raw `shap_values` matrix (for an
    `.npz` export), and `dependence_plot_paths` (empty unless
    `dependence_plot_dir` is given, since rendering plots is the only step
    here with a side effect - tests exercising the rest of this bundle don't
    need to write files).
    """
    shap_values = compute_tree_shap(model, X)
    groups = split_shap_by_group(shap_values, embedding_dim, physical_dim)

    dependence_plot_paths = {}
    if dependence_plot_dir is not None:
        X_physical = X[:, embedding_dim:]
        dependence_plot_paths = save_dependence_plots(
            X_physical, groups["physical"], dependence_plot_dir
        )

    return {
        "top_embedding_features": top_k_embedding_features(
            groups["embedding"], k=top_k
        ),
        "physical_feature_shap": physical_feature_shap_summary(groups["physical"]),
        "physical_contribution_ratio": aggregate_physical_contribution_ratio(
            shap_values, embedding_dim
        ),
        "shap_values": shap_values,
        "dependence_plot_paths": dependence_plot_paths,
    }


In [ ]:
%%writefile case_study_selection.py
"""Case Study selection (Step 5, issue #17; Reverse Discordant, issue #18).

Selects the 15 representative Testset samples used for deep per-sample
explainability bundling, following the residual-quantile rule documented in
CONTEXT.md / plan_realize.md Step 5:

- **Primary Discordant** (preferred): Model A predicts poorly, Model B
  predicts well (`e_A >= Q75(e_A) AND e_B <= Q25(e_B)`). Top 5 by
  discordance gap (`e_A - e_B`, descending) - the largest "A got this wrong,
  B got this right" gap first.
- **Secondary Discordant** (fallback only): used to fill remaining
  Discordant-family slots *only* when Primary yields fewer than 5
  (`(e_A - e_B) / e_A >= 0.5 AND e_A >= Median(e_A)`), ranked by relative
  gap descending. Never overlaps with the Primary selection.
- **Reverse Primary Discordant** (issue #18; preferred): the mirror image -
  Model B predicts poorly, Model A predicts well (`e_B >= Q75(e_B) AND
  e_A <= Q25(e_A)`). Top 5 by reverse gap (`e_B - e_A`, descending) - the
  largest "B regressed relative to A" gap first. This is a separate 5-slot
  budget from the forward Discordant family (not a subset of it), since the
  reverse population is comparably sized to the forward one (see issue #18)
  and would starve one direction if the two shared a single 5-slot budget.
- **Reverse Secondary Discordant** (fallback only): fills remaining
  Reverse-family slots *only* when Reverse Primary yields fewer than 5
  (`(e_B - e_A) / e_B >= 0.5 AND e_B >= Median(e_B)`), ranked by relative
  gap descending. Never overlaps with the Reverse Primary selection, nor
  with anything already selected by the forward Discordant family.
- **Concordant**: both models predict well (`e_A <= Q25(e_A) AND
  e_B <= Q25(e_B)`). Top 5 by combined error (`e_A + e_B`, ascending) - the
  best joint accuracy first.

Total Case Study count is capped at 15 (5 Discordant-family + 5
Reverse-Discordant-family + 5 Concordant); any group can come up short of
its target if too few samples qualify - this is not itself an error, just a
smaller-than-usual export.

Ties in every ranking break by ascending array index, for a fully
deterministic selection given the same `e_a`/`e_b` input.
"""

import numpy as np

PRIMARY_TARGET = 5
REVERSE_TARGET = 5
CONCORDANT_TARGET = 5
SECONDARY_RELATIVE_GAP_THRESHOLD = 0.5

PRIMARY_DISCORDANT = "Primary Discordant"
SECONDARY_DISCORDANT = "Secondary Discordant"
REVERSE_PRIMARY_DISCORDANT = "Reverse Primary Discordant"
REVERSE_SECONDARY_DISCORDANT = "Reverse Secondary Discordant"
CONCORDANT = "Concordant"


def _rank(indices, key) -> list[int]:
    """Stable-ish ranking: descending/ascending `key`, ties broken by ascending index."""
    return sorted(indices, key=key)


def _select_discordant_family(
    e_hi: np.ndarray, e_lo: np.ndarray, target: int, exclude: set
) -> tuple[list[int], list[int]]:
    """Top-`target` gap-descending selection with a relative-gap fallback.

    Shared by both directions of the Discordant family (Primary/Secondary
    Discordant calls this with `e_hi=e_a, e_lo=e_b`; Reverse Primary/
    Secondary Discordant calls it with `e_hi=e_b, e_lo=e_a`):

    - Primary condition: `e_hi >= Q75(e_hi) AND e_lo <= Q25(e_lo)`, ranked by
      `(e_hi - e_lo)` descending.
    - Secondary fallback (only fills what Primary leaves short):
      `(e_hi - e_lo) / e_hi >= SECONDARY_RELATIVE_GAP_THRESHOLD AND
      e_hi >= Median(e_hi)`, ranked by relative gap descending. Never
      overlaps with this family's own Primary selection.

    `exclude` is a set of indices this family must never select (already
    claimed by another family) - applied to both the Primary and Secondary
    candidate pools.

    Returns `(primary_selected, secondary_selected)` index lists.
    """
    q75_hi = np.quantile(e_hi, 0.75)
    q25_lo = np.quantile(e_lo, 0.25)
    median_hi = np.median(e_hi)

    primary_mask = (e_hi >= q75_hi) & (e_lo <= q25_lo)
    primary_candidates = [i for i in np.flatnonzero(primary_mask) if i not in exclude]
    gap = e_hi - e_lo
    primary_selected = _rank(primary_candidates, key=lambda i: (-gap[i], i))[:target]

    secondary_selected: list[int] = []
    remaining = target - len(primary_selected)
    if remaining > 0:
        primary_selected_set = set(primary_selected)
        with np.errstate(divide="ignore", invalid="ignore"):
            relative_gap = (e_hi - e_lo) / e_hi
        secondary_mask = (relative_gap >= SECONDARY_RELATIVE_GAP_THRESHOLD) & (
            e_hi >= median_hi
        )
        secondary_candidates = [
            i
            for i in np.flatnonzero(secondary_mask)
            if i not in primary_selected_set and i not in exclude
        ]
        secondary_selected = _rank(
            secondary_candidates, key=lambda i: (-relative_gap[i], i)
        )[:remaining]

    return primary_selected, secondary_selected


def _entries_for(selected: list[int], case_type: str, id_prefix: str) -> list[dict]:
    return [
        {"index": int(idx), "case_type": case_type, "case_id": f"{id_prefix}{rank:02d}"}
        for rank, idx in enumerate(selected, start=1)
    ]


def select_case_studies(e_a: np.ndarray, e_b: np.ndarray) -> list[dict]:
    """Select up to 15 Case Study indices per the documented residual-quantile rule.

    `e_a`/`e_b` are 1D arrays of per-sample absolute error for Model A / Model
    B, aligned by index (index `i` is Testset `sample_id` `i`, since
    `test_metadata.csv` assigns `sample_id` by contiguous row order).

    Returns a list of dicts (`index`, `case_type`, `case_id`), ordered
    Primary Discordant, then Secondary Discordant (if any), then Reverse
    Primary Discordant, then Reverse Secondary Discordant (if any), then
    Concordant - at most `PRIMARY_TARGET + REVERSE_TARGET + CONCORDANT_TARGET`
    (15) entries total.
    """
    e_a = np.asarray(e_a, dtype=np.float64)
    e_b = np.asarray(e_b, dtype=np.float64)
    if e_a.shape != e_b.shape:
        raise ValueError(
            f"e_a and e_b must have the same shape, got {e_a.shape} vs {e_b.shape}"
        )
    if e_a.ndim != 1:
        raise ValueError(f"e_a/e_b must be 1D, got ndim={e_a.ndim}")

    primary_selected, secondary_selected = _select_discordant_family(
        e_a, e_b, PRIMARY_TARGET, exclude=set()
    )
    forward_selected_set = set(primary_selected) | set(secondary_selected)

    reverse_primary_selected, reverse_secondary_selected = _select_discordant_family(
        e_b, e_a, REVERSE_TARGET, exclude=forward_selected_set
    )

    # --- Concordant ---
    q25_a = np.quantile(e_a, 0.25)
    q25_b = np.quantile(e_b, 0.25)
    concordant_mask = (e_a <= q25_a) & (e_b <= q25_b)
    concordant_candidates = np.flatnonzero(concordant_mask)
    combined_error = e_a + e_b
    concordant_selected = _rank(
        concordant_candidates.tolist(), key=lambda i: (combined_error[i], i)
    )[:CONCORDANT_TARGET]

    return (
        _entries_for(primary_selected, PRIMARY_DISCORDANT, "DISCORDANT_P")
        + _entries_for(secondary_selected, SECONDARY_DISCORDANT, "DISCORDANT_S")
        + _entries_for(
            reverse_primary_selected, REVERSE_PRIMARY_DISCORDANT, "DISCORDANT_R"
        )
        + _entries_for(
            reverse_secondary_selected, REVERSE_SECONDARY_DISCORDANT, "DISCORDANT_RS"
        )
        + _entries_for(concordant_selected, CONCORDANT, "CONCORDANT_C")
    )


In [ ]:
%%writefile analysis_export.py
"""Standardized artifact export: Case Study bundling + schema assembly (Step 5, issue #17).

This is the pipeline's final assembly step (see plan_realize.md section 4 /
issue #9's Implementation Decisions): it does not compute new explainability
signal itself (that's Steps 2-4, issues #11-#16) - it selects the 15
representative Case Study samples (`case_study_selection.py`), bundles each
with its per-sample detail, and assembles the standard two-artifact export:

- `model_analysis_arrays.npz`: every large multi-dimensional tensor
  (ISM/mismatch/attention/SHAP arrays for the full Testset).
- `model_analysis_summary.json`: metadata, `global_evaluation`,
  `complex_mismatch_metadata`, and the `case_studies` list - no large arrays,
  per issue #9's AC24/AC25.
"""

import json

import numpy as np

# Order matches Model_B_Predictor.compute_physical_features's hstack order
# (mfe, dg, tm, gc) - see model_b_wrapper.py / shap_analysis.py.
PHYSICAL_FEATURE_KEYS = ("mfe", "dg", "tm", "gc")

PHYSICAL_SHAP_POSITIVE_INTERPRETATION = (
    "Physical features' SHAP contribution shifted Model B's prediction "
    "toward the ground truth."
)
PHYSICAL_SHAP_NEGATIVE_INTERPRETATION = (
    "Physical features' SHAP contribution shifted Model B's prediction "
    "away from the ground truth."
)

REQUIRED_TOP_LEVEL_KEYS = (
    "metadata",
    "global_evaluation",
    "complex_mismatch_metadata",
    "case_studies",
)

MAX_CASE_STUDIES = 15

# Large multi-dimensional arrays that must live only in the .npz - never
# embedded as a JSON key anywhere in the summary (issue #9 AC24/#17 AC5).
LARGE_ARRAY_KEYS = frozenset(
    {
        "ism_delta",
        "ism_delta_model_a",
        "ism_delta_model_b",
        "mismatch_single_delta",
        "mismatch_single_delta_model_a",
        "mismatch_single_delta_model_b",
        "mismatch_single_min",
        "mismatch_single_min_model_a",
        "mismatch_single_min_model_b",
        "mismatch_complex",
        "mismatch_complex_model_a",
        "mismatch_complex_model_b",
        "attention_rollout",
        "shap_values",
    }
)


def to_jsonable(value):
    """Recursively convert numpy scalars/arrays to plain Python/JSON-safe types."""
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {k: to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    return value


def physical_shap_interpretation(
    true_score: float, pred_b: float, physical_shap_values
) -> str:
    """Direction-aware interpretation of a sample's physical-feature SHAP contribution.

    Rather than a fixed narrative string, this compares Model B's actual
    error against the counterfactual error it would have had without the
    physical features' contribution (`pred_b` minus the sum of their SHAP
    values, since SHAP contributions are additive around the model's
    expected value): if removing the physical contribution would have
    pushed the prediction further from `true_score`, the physical features
    are reported as having moved the prediction *toward* the truth, and
    vice versa.
    """
    physical_shap_sum = float(np.sum(physical_shap_values))
    pred_without_physical = pred_b - physical_shap_sum
    error_with_physical = abs(true_score - pred_b)
    error_without_physical = abs(true_score - pred_without_physical)

    if error_with_physical <= error_without_physical:
        return PHYSICAL_SHAP_POSITIVE_INTERPRETATION
    return PHYSICAL_SHAP_NEGATIVE_INTERPRETATION


def build_case_study_entry(
    case: dict,
    *,
    sample_id: int,
    original_id: int,
    sequence: str,
    true_score: float,
    pred_a: float,
    error_a: float,
    pred_b: float,
    error_b: float,
    physical_values: dict,
    physical_shap: dict,
    ig_model_a_norm_attr_36bp,
    ig_model_b_norm_attr_36bp,
    attention_rollout_info: dict,
) -> dict:
    """Bundle one selected Case Study's full per-sample explainability detail.

    `case` is one entry from `case_study_selection.select_case_studies`
    (`case_id`/`case_type`); everything else is that same sample's already-
    computed data. `physical_values`/`physical_shap` are dicts keyed by
    `PHYSICAL_FEATURE_KEYS`. Large arrays (the attention matrix itself) are
    deliberately not embedded here - only `attention_rollout_info`'s
    dimensions/note - per issue #9 AC24/#17 AC5; the IG attribution arrays
    are a fixed, small 36-length vector so they're embedded directly, same
    as plan_realize.md section 4's schema example.
    """
    interpretation = physical_shap_interpretation(
        true_score, pred_b, [physical_shap[k] for k in PHYSICAL_FEATURE_KEYS]
    )

    return {
        "case_id": case["case_id"],
        "sample_id": int(sample_id),
        "original_id": int(original_id),
        "sequence": sequence,
        "true_score": float(true_score),
        "model_a": {"pred_raw": float(pred_a), "error": float(error_a)},
        "model_b": {"pred_raw": float(pred_b), "error": float(error_b)},
        "case_type": case["case_type"],
        "per_sample_physical_values": {
            k: float(physical_values[k]) for k in PHYSICAL_FEATURE_KEYS
        },
        "per_sample_physical_shap": {
            **{k: float(physical_shap[k]) for k in PHYSICAL_FEATURE_KEYS},
            "interpretation": interpretation,
        },
        "integrated_gradients": {
            "model_a_norm_attr_36bp": to_jsonable(
                np.asarray(ig_model_a_norm_attr_36bp)
            ),
            "model_b_phase4_projected_norm_attr_36bp": to_jsonable(
                np.asarray(ig_model_b_norm_attr_36bp)
            ),
        },
        "attention_rollout": dict(attention_rollout_info),
    }


def assemble_case_studies(
    selection: list[dict],
    *,
    meta_df,
    pred_a: np.ndarray,
    error_a: np.ndarray,
    pred_b: np.ndarray,
    error_b: np.ndarray,
    physical_values_by_index: dict,
    physical_shap_by_index: dict,
    ig_by_index: dict,
    attention_rollout_info: dict,
) -> list[dict]:
    """Build the full `case_studies` list from a `select_case_studies` selection.

    `pred_a`/`error_a`/`pred_b`/`error_b` are full-Testset-length arrays
    (residual/quantile computation over the whole Testset is what selection
    itself needed), indexed by each case's `index`. `physical_values_by_index`
    /`physical_shap_by_index`/`ig_by_index` instead map a selected sample's
    index directly to its already-computed data (a 4-value
    `PHYSICAL_FEATURE_KEYS`-ordered array-like, and an
    `{"model_a_norm_attr_36bp": ..., "model_b_phase4_projected_norm_attr_36bp": ...}`
    dict, respectively) - physical values/SHAP/Integrated Gradients are only
    ever computed for the (at most 15) selected Case Study samples, never
    the full Testset, since that's all this export needs (issue #17's AC3
    scopes IG/attention detail to the Case Studies themselves), so there is
    no full-population array to index into for these three.
    """
    entries = []
    for case in selection:
        idx = case["index"]
        row = meta_df.iloc[idx]
        entries.append(
            build_case_study_entry(
                case,
                sample_id=row["sample_id"],
                original_id=row["original_id"],
                sequence=row["sequence"],
                true_score=row["true_score"],
                pred_a=pred_a[idx],
                error_a=error_a[idx],
                pred_b=pred_b[idx],
                error_b=error_b[idx],
                physical_values=dict(
                    zip(PHYSICAL_FEATURE_KEYS, physical_values_by_index[idx])
                ),
                physical_shap=dict(
                    zip(PHYSICAL_FEATURE_KEYS, physical_shap_by_index[idx])
                ),
                ig_model_a_norm_attr_36bp=ig_by_index[idx]["model_a_norm_attr_36bp"],
                ig_model_b_norm_attr_36bp=ig_by_index[idx][
                    "model_b_phase4_projected_norm_attr_36bp"
                ],
                attention_rollout_info=attention_rollout_info,
            )
        )
    return entries


def build_model_analysis_summary(
    metadata: dict,
    global_evaluation: dict,
    complex_mismatch_metadata: list,
    case_studies: list,
) -> dict:
    """Assemble the standard `model_analysis_summary.json` top-level dict."""
    return {
        "metadata": metadata,
        "global_evaluation": global_evaluation,
        "complex_mismatch_metadata": complex_mismatch_metadata,
        "case_studies": case_studies,
    }


def _is_leaked_array_value(value) -> bool:
    """Whether `value` looks like actual leaked tensor data, not descriptive metadata.

    An `ndarray` is always a leak. A `list`/`tuple` is only a leak if it
    isn't purely made of strings - real array data (or its post-
    `to_jsonable` nested-list form) is numbers/further nesting, whereas
    `metadata.array_axis_definitions` legitimately holds plain lists of
    human-readable axis-label strings under these same key names (per
    plan_realize.md section 4's schema) and must not be flagged.
    """
    if isinstance(value, np.ndarray):
        return True
    if isinstance(value, (list, tuple)):
        return not all(isinstance(item, str) for item in value)
    return False


def _find_large_array_keys(obj) -> set:
    """Recursively collect any `LARGE_ARRAY_KEYS` name whose *array-shaped* value appears anywhere in `obj`.

    A `LARGE_ARRAY_KEYS` name used as a key is only a violation when its
    value actually looks like leaked array data (see
    `_is_leaked_array_value`). It is legitimate for a Case Study entry's
    small `attention_rollout` *metadata* dict (dims + note, no matrix data -
    see `build_case_study_entry`) and for `metadata.array_axis_definitions`'s
    axis-label-string lists to reuse these same names, so neither is itself
    a violation.
    """
    found = set()
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key in LARGE_ARRAY_KEYS and _is_leaked_array_value(value):
                found.add(key)
            found |= _find_large_array_keys(value)
    elif isinstance(obj, (list, tuple)):
        for item in obj:
            found |= _find_large_array_keys(item)
    return found


def validate_summary_schema(summary: dict) -> None:
    """Validate `summary` against the Section-4 export schema (issue #17 AC4/AC5).

    Checks: all required top-level keys present, `case_studies` count capped
    at `MAX_CASE_STUDIES`, no `LARGE_ARRAY_KEYS` name leaked into the JSON
    (they must live only in the `.npz`), and the summary is actually
    JSON-serializable (numpy scalars/arrays would raise `TypeError` here
    instead of silently producing malformed JSON).
    """
    missing = [key for key in REQUIRED_TOP_LEVEL_KEYS if key not in summary]
    if missing:
        raise ValueError(
            f"model_analysis_summary.json missing required keys: {missing}"
        )

    if len(summary["case_studies"]) > MAX_CASE_STUDIES:
        raise ValueError(
            f"case_studies has {len(summary['case_studies'])} entries, "
            f"exceeding the cap of {MAX_CASE_STUDIES}"
        )

    leaked = _find_large_array_keys(summary)
    if leaked:
        raise ValueError(
            f"Large array key(s) {sorted(leaked)} must not appear in "
            "model_analysis_summary.json - they belong only in model_analysis_arrays.npz"
        )

    json.dumps(summary)  # raises TypeError on any un-jsonified numpy leakage


def export_artifacts(
    summary: dict,
    arrays: dict,
    json_path: str,
    npz_path: str,
) -> None:
    """Validate and write the two standard export artifacts.

    `arrays` is everything destined for `model_analysis_arrays.npz` (full
    Testset ISM/mismatch/attention/SHAP tensors); `summary` is the
    JSON-only metadata + Case Studies dict. Validation runs before either
    file is written, so a schema violation never leaves a half-written pair
    on disk.
    """
    validate_summary_schema(summary)

    jsonable_summary = to_jsonable(summary)
    with open(json_path, "w") as f:
        json.dump(jsonable_summary, f, indent=2)

    np.savez(npz_path, **arrays)


In [ ]:
%%writefile step5_export.py
"""Step 5 runner: Case Study selection + standardized artifact export (issue #17).

Assembles the final `model_analysis_arrays.npz` / `model_analysis_summary.json`
pair from this Testset's predictions plus every prior step's own output
artifact (issues #11-#16). Requires (in the working directory):

- `test_metadata.csv` (issue #1, `test_dataset_verifier.py`)
- `ism_delta.npz` (issue #11, `step3_ism.py`)
- `mismatch_delta.npz` + `mismatch_complex_metadata.json` (issue #12, `step3_mismatch.py`)
- `ablation_results.json` (issue #13, `step4_ablation.py`)
- `shap_results.json` + `shap_values.npz` (issue #14, `step4_shap.py`)
- `attention_rollout_matrix.npz` + `attention_rollout_summary.json` (issue #16, `step4_attention_rollout.py`)

Integrated Gradients (issue #15) has no standalone runner/artifact of its
own - it is only ever needed for the (at most 15) selected Case Study
samples here, so this script computes it directly for just those sequences
rather than requiring a full-Testset IG artifact no other step produces.
"""

import hashlib
import json
import os

import numpy as np
import pandas as pd
from analysis_export import (
    assemble_case_studies,
    build_model_analysis_summary,
    export_artifacts,
)
from case_study_selection import select_case_studies
from integrated_gradients import (
    integrated_gradients_model_a,
    integrated_gradients_model_b_tokens,
    l1_normalize,
    project_model_b_attributions_to_nucleotides,
)
from model_a_wrapper import Model_A_Predictor
from model_b_wrapper import Model_B_Predictor
from model_b_xai_wrapper import Model_B_XAIPredictor
from shap_analysis import (
    EMBEDDING_DIM,
    PHYSICAL_DIM,
    compute_tree_shap,
    split_shap_by_group,
)

TEST_METADATA_PATH = "test_metadata.csv"
MODEL_A_WEIGHT_PATH = "best_model_fold1.pth"
NT_MODEL_DIR = "./NT_sacas9_fintuned_model"
XGB_MODEL_PATH = "hybrid_xgb_model.json"

ISM_ARRAYS_PATH = "ism_delta.npz"
MISMATCH_ARRAYS_PATH = "mismatch_delta.npz"
MISMATCH_COMPLEX_METADATA_PATH = "mismatch_complex_metadata.json"
ABLATION_RESULTS_PATH = "ablation_results.json"
SHAP_RESULTS_PATH = "shap_results.json"
SHAP_VALUES_PATH = "shap_values.npz"
ATTENTION_ROLLOUT_ARRAYS_PATH = "attention_rollout_matrix.npz"
ATTENTION_ROLLOUT_SUMMARY_PATH = "attention_rollout_summary.json"

OUTPUT_JSON_PATH = "model_analysis_summary.json"
OUTPUT_NPZ_PATH = "model_analysis_arrays.npz"

# Derived from the same constants integrated_gradients.py/shap_analysis.py
# already define for these dimensions, rather than re-hardcoding them here
# and risking the two definitions drifting apart.
MODEL_B_ARCHITECTURE = {
    "tokenizer": "6-mer",
    "tokens": Model_B_XAIPredictor.EXPECTED_SEQ_LEN,
    "hidden_dim": Model_B_XAIPredictor.EXPECTED_HIDDEN_DIM,
    "flattened_dim": EMBEDDING_DIM,
    "physical_dim": PHYSICAL_DIM,
    "total_feature_dim": EMBEDDING_DIM + PHYSICAL_DIM,
}

ARRAY_AXIS_DEFINITIONS = {
    "ism_delta_model_a": [
        "sample_id (0~513)",
        "position (0~35)",
        "alternative_base (lexicographical 3 bases)",
    ],
    "ism_delta_model_b": [
        "sample_id (0~513)",
        "position (0~35)",
        "alternative_base (lexicographical 3 bases)",
    ],
    "mismatch_single_delta_model_a": [
        "sample_id (0~513)",
        "position (0~35)",
        "alternative_base (3 bases)",
    ],
    "mismatch_single_delta_model_b": [
        "sample_id (0~513)",
        "position (0~35)",
        "alternative_base (3 bases)",
    ],
    "mismatch_single_min_model_a": ["sample_id (0~513)", "position (0~35)"],
    "mismatch_single_min_model_b": ["sample_id (0~513)", "position (0~35)"],
    "mismatch_complex_model_a": ["sample_id (0~513)", "scenario_id (0~14)"],
    "mismatch_complex_model_b": ["sample_id (0~513)", "scenario_id (0~14)"],
    "shap_values": [
        "sample_id (0~513)",
        "feature_index (0~8963, embedding then physical)",
    ],
    "attention_rollout": [
        "sample_id (0~513)",
        "row_position (0~35)",
        "column_position (0~35)",
    ],
}


def _require(path: str, produced_by: str) -> str:
    # Same "{path} not found. {script}를 먼저 실행해 주세요." phrasing every other
    # stepN_*.py runner in this repo uses (step1_verify.py, step3_ism.py, ...).
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. {produced_by}를 먼저 실행해 주세요."
        )
    return path


def _load_npz_as_dict(path: str) -> dict:
    with np.load(path) as data:
        return {key: data[key] for key in data.files}


def _dataset_sha256(sequences: pd.Series) -> str:
    # Same whole-column hashing convention as test_dataset_verifier.py, so
    # this checksum is directly comparable to the one printed at Testset
    # isolation time.
    return hashlib.sha256(pd.util.hash_pandas_object(sequences).values).hexdigest()


def build_metadata(meta_df: pd.DataFrame) -> dict:
    return {
        "testset_version": "SaCas9_v4_independent_10pct",
        "total_samples": len(meta_df),
        "sha256_checksum": _dataset_sha256(meta_df["sequence"]),
        "model_b_architecture": MODEL_B_ARCHITECTURE,
        "array_axis_definitions": ARRAY_AXIS_DEFINITIONS,
    }


def build_global_evaluation(ablation_results: dict, shap_results: dict) -> dict:
    # plan_realize.md section 4 nests `paired_bootstrap_95ci` inside
    # `ablation_results`, but `ablation.py`/`step4_ablation.py` (issue #13)
    # write it as a sibling key in `ablation_results.json` - re-nest here so
    # the final export matches the documented schema shape.
    return {
        "ablation_results": {
            **ablation_results["ablation_results"],
            "paired_bootstrap_95ci": ablation_results["paired_bootstrap_95ci"],
        },
        "aggregate_physical_shap_contribution_ratio": shap_results[
            "physical_contribution_ratio"
        ],
    }


def compute_case_study_ig(
    predictor_a: Model_A_Predictor,
    xai_predictor_b: Model_B_XAIPredictor,
    selection: list[dict],
    sequences: list[str],
) -> dict:
    """Integrated Gradients for just the selected Case Study sequences (see module docstring)."""
    selected_indices = [case["index"] for case in selection]
    selected_sequences = [sequences[i] for i in selected_indices]

    ig_a_norm = l1_normalize(
        integrated_gradients_model_a(predictor_a, selected_sequences)
    )
    ig_b_tokens = integrated_gradients_model_b_tokens(
        xai_predictor_b, selected_sequences
    )
    ig_b_norm = l1_normalize(project_model_b_attributions_to_nucleotides(ig_b_tokens))

    return {
        idx: {
            "model_a_norm_attr_36bp": ig_a_norm[j],
            "model_b_phase4_projected_norm_attr_36bp": ig_b_norm[j],
        }
        for j, idx in enumerate(selected_indices)
    }


def compute_case_study_physical_shap(
    predictor_b: Model_B_Predictor,
    selection: list[dict],
    sequences: list[str],
    shap_values_full: np.ndarray | None,
) -> np.ndarray:
    """Per-selected-sample physical-feature SHAP, shape `(len(selection), 4)`.

    Reuses `shap_values.npz`'s full `(514, 8964)` matrix when available
    (the same values Tree SHAP over the whole Testset already produced);
    otherwise computes Tree SHAP directly for just the selected sequences -
    cheap at 10 samples, unlike the full 514-sample run issue #14 owns.
    """
    selected_indices = [case["index"] for case in selection]

    if shap_values_full is not None:
        physical_shap = split_shap_by_group(shap_values_full)["physical"]
        return physical_shap[selected_indices]

    selected_sequences = [sequences[i] for i in selected_indices]
    embeddings = predictor_b.extract_nt_embeddings(selected_sequences)
    physical = predictor_b.compute_physical_features(selected_sequences)
    X_selected = np.hstack([embeddings, physical])
    shap_values = compute_tree_shap(predictor_b.xgb_model, X_selected)
    return split_shap_by_group(shap_values, physical_dim=PHYSICAL_DIM)["physical"]


def main():
    print("=== [Step 5] Case Study Selection + Standardized Artifact Export ===")

    _require(TEST_METADATA_PATH, "test_dataset_verifier.py")
    meta_df = pd.read_csv(TEST_METADATA_PATH)
    sequences = meta_df["sequence"].tolist()
    true_scores = meta_df["true_score"].to_numpy()
    print(f"Loaded testset: {len(sequences)} samples.")

    print("\nInitializing Model A / Model B predictors...")
    predictor_a = Model_A_Predictor(weight_path=MODEL_A_WEIGHT_PATH)
    predictor_b = Model_B_Predictor(
        nt_model_dir=NT_MODEL_DIR, xgb_model_path=XGB_MODEL_PATH
    )

    print("\nScoring the Testset with both models (no cross-model rescaling)...")
    pred_a = np.asarray(predictor_a.predict(sequences), dtype=np.float64)
    pred_b = np.asarray(predictor_b.predict(sequences), dtype=np.float64)
    error_a = np.abs(true_scores - pred_a)
    error_b = np.abs(true_scores - pred_b)

    print(
        "\nSelecting Case Studies (Primary -> Secondary fallback -> "
        "Reverse Primary -> Reverse Secondary fallback -> Concordant)..."
    )
    selection = select_case_studies(error_a, error_b)
    print(
        f" - Selected {len(selection)} Case Studies: "
        f"{[c['case_id'] for c in selection]}"
    )

    print("\nLoading prior steps' precomputed artifacts...")
    ism_arrays = _load_npz_as_dict(_require(ISM_ARRAYS_PATH, "step3_ism.py"))
    mismatch_arrays = _load_npz_as_dict(
        _require(MISMATCH_ARRAYS_PATH, "step3_mismatch.py")
    )
    with open(_require(MISMATCH_COMPLEX_METADATA_PATH, "step3_mismatch.py")) as f:
        complex_mismatch_metadata = json.load(f)
    with open(_require(ABLATION_RESULTS_PATH, "step4_ablation.py")) as f:
        ablation_results = json.load(f)
    with open(_require(SHAP_RESULTS_PATH, "step4_shap.py")) as f:
        shap_results = json.load(f)
    shap_values_full = None
    if os.path.exists(SHAP_VALUES_PATH):
        shap_values_full = _load_npz_as_dict(SHAP_VALUES_PATH)["shap_values"]
    attention_arrays = _load_npz_as_dict(
        _require(ATTENTION_ROLLOUT_ARRAYS_PATH, "step4_attention_rollout.py")
    )
    with open(
        _require(ATTENTION_ROLLOUT_SUMMARY_PATH, "step4_attention_rollout.py")
    ) as f:
        attention_rollout_summary = json.load(f)

    print(
        "\nComputing per-Case-Study physical values, physical SHAP, and Integrated Gradients..."
    )
    xai_predictor_b = Model_B_XAIPredictor(nt_model_dir=NT_MODEL_DIR)
    selected_indices = [case["index"] for case in selection]
    physical_values_selected = predictor_b.compute_physical_features(
        [sequences[i] for i in selected_indices]
    )
    physical_values_by_index = dict(zip(selected_indices, physical_values_selected))

    physical_shap_selected = compute_case_study_physical_shap(
        predictor_b, selection, sequences, shap_values_full
    )
    physical_shap_by_index = dict(zip(selected_indices, physical_shap_selected))

    ig_by_index = compute_case_study_ig(
        predictor_a, xai_predictor_b, selection, sequences
    )

    attention_rollout_info = {
        "token_matrix_dim": attention_rollout_summary["token_matrix_dim"],
        "projected_nucleotide_matrix_dim": attention_rollout_summary[
            "projected_nucleotide_matrix_dim"
        ],
        "note": attention_rollout_summary["note"],
    }

    case_studies = assemble_case_studies(
        selection,
        meta_df=meta_df,
        pred_a=pred_a,
        error_a=error_a,
        pred_b=pred_b,
        error_b=error_b,
        physical_values_by_index=physical_values_by_index,
        physical_shap_by_index=physical_shap_by_index,
        ig_by_index=ig_by_index,
        attention_rollout_info=attention_rollout_info,
    )

    summary = build_model_analysis_summary(
        metadata=build_metadata(meta_df),
        global_evaluation=build_global_evaluation(ablation_results, shap_results),
        complex_mismatch_metadata=complex_mismatch_metadata,
        case_studies=case_studies,
    )

    arrays = {**ism_arrays, **mismatch_arrays, **attention_arrays}
    if shap_values_full is not None:
        arrays["shap_values"] = shap_values_full

    print(f"\nWriting {OUTPUT_JSON_PATH} and {OUTPUT_NPZ_PATH}...")
    export_artifacts(summary, arrays, OUTPUT_JSON_PATH, OUTPUT_NPZ_PATH)
    print("Done.")

    print("\n=== Step 5 (Case Study Export) Execution Completed ===")


if __name__ == "__main__":
    main()


## 5. Confirm GPU is available

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected - Runtime > Change runtime type > GPU")

## 6. Run Step 5: Case Study selection + standardized artifact export

Writes `model_analysis_summary.json` and `model_analysis_arrays.npz`
directly into `DRIVE_DIR` (already the working directory).


In [ ]:
!python step5_export.py

## 7. Spot-check: Model B Integrated Gradients is no longer all-zero

Directly re-checks the exact field that was originally reported as all
zero, on the freshly regenerated file.


In [ ]:
import json

with open("model_analysis_summary.json") as f:
    summary = json.load(f)

for case in summary["case_studies"]:
    vec = case["integrated_gradients"]["model_b_phase4_projected_norm_attr_36bp"]
    nonzero = any(v != 0 for v in vec)
    print(f"{case['case_id']}: any nonzero = {nonzero}")

## 8. Download the two output files

Copy `model_analysis_summary.json` and `model_analysis_arrays.npz` from
`DRIVE_DIR` back into your local `xai_test/` repo checkout, replacing the
existing copies - same pattern as `colab_shap.ipynb`'s results.
